In [0]:
# ======================= INSTALL =======================

!apt-get update
!apt-get install -y tesseract-ocr poppler-utils libreoffice

%pip install PyMuPDF python-docx pytesseract pillow pdf2image
dbutils.library.restartPython()

In [0]:
# ======================= IMPORTS =======================
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import col, length
import fitz
from docx import Document
import os
import subprocess
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

print("JOB STARTED (TESSERACT OCR)")

# ======================= READ FILES =======================
pdf_df = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load("dbfs:/FileStore/noc_pm_pdfs")
    .select("path", "content")
)
print("Files read from DBFS")

# ======================= LOAD PROCESSED =======================
try:
    processed_df = (
        spark.table("noc_pm_documents_tessaract")
        .filter(col("final_text").isNotNull() & (length(col("final_text")) > 0))
        .select("path")
        .distinct()
    )
    print(f"Already processed files: {processed_df.count()}")
except Exception:
    processed_df = spark.createDataFrame([], "path STRING")
    print("Table noc_pm_documents_tessaract does not exist")

# ======================= UNPROCESSED =======================
unprocessed_df = pdf_df.join(
    processed_df,
    on="path",
    how="left_anti"
)

print(f"Unprocessed files count: {unprocessed_df.count()}")

# ======================= OCR HELPERS =======================
TMP_DIR = "/tmp/tesseract"
os.makedirs(TMP_DIR, exist_ok=True)

def convert_to_pdf(binary, path):
    ext = os.path.splitext(path.lower())[1]
    base = os.path.basename(path)
    src = f"{TMP_DIR}/{base}"
    pdf = f"{TMP_DIR}/{base}.pdf"

    with open(src, "wb") as f:
        f.write(binary)

    if ext == ".pdf":
        return src

    subprocess.run(
        [
            "libreoffice",
            "--headless",
            "--convert-to",
            "pdf",
            "--outdir",
            TMP_DIR,
            src
        ],
        check=True
    )
    return pdf

def ocr_tesseract(binary, path):
    print("[TESSERACT] OCR started")
    try:
        pdf_path = convert_to_pdf(binary, path)

        images = convert_from_path(
            pdf_path,
            dpi=300,
            output_folder=TMP_DIR
        )

        text = []
        for img in images:
            text.append(
                pytesseract.image_to_string(img, lang="eng")
            )

        final_text = "\n".join(text)
        print(f"[TESSERACT] Success ({len(final_text)} chars)")
        return final_text

    except Exception as e:
        print(f"[TESSERACT ERROR] {path} -> {e}")
        return ""

# ======================= TEXT EXTRACTORS =======================
def extract_text_from_pdf(binary, path):
    print("[FITZ] Extracting Text")
    try:
        doc = fitz.open(stream=binary, filetype="pdf")
        text = "\n".join(page.get_text() for page in doc)
        print(f"[FITZ] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[FITZ ERROR] {path} -> {e}")
        return ""

def extract_text_from_docx(binary, path):
    print("[DOCX] Extracting Text")
    try:
        tmp_path = f"/tmp/{os.path.basename(path)}"
        with open(tmp_path, "wb") as f:
            f.write(binary)

        doc = Document(tmp_path)
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
        print(f"[DOCX] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[DOCX ERROR] {path} -> {e}")
        return ""

def extract_text_from_plain(binary, path):
    try:
        text = binary.decode("utf-8", errors="ignore")
        print(f"[PLAIN] Success ({len(text)} chars)")
        return text
    except Exception:
        return ""

def extract_text(binary, path):
    ext = os.path.splitext(path.lower())[1]

    if ext == ".pdf":
        text = extract_text_from_pdf(binary, path)

    elif ext == ".docx":
        text = extract_text_from_docx(binary, path)

    elif ext == ".doc":
        print("[DOC] Legacy .doc → Tesseract OCR")
        return ocr_tesseract(binary, path)

    elif ext in {".txt", ".csv", ".md", ".rtf"}:
        text = extract_text_from_plain(binary, path)

    else:
        print(f"[INFO] Unsupported format {ext} → OCR")
        return ocr_tesseract(binary, path)

    if not text or len(text.strip()) < 200:
        print("[FALLBACK] Text too small → OCR")
        return ocr_tesseract(binary, path)

    return text

# ======================= WRITE =======================
schema = StructType([
    StructField("path", StringType(), True),
    StructField("final_text", StringType(), True),
    StructField("is_scanned", BooleanType(), True)
])

rows = list(unprocessed_df.toLocalIterator())

for row in rows[:40]:
    path = row["path"]
    binary = row["content"]

    try:
        final_text = extract_text(binary, path)
        is_scanned = len(final_text.strip()) < 200

        result_df = spark.createDataFrame(
            [(path, final_text, is_scanned)],
            schema
        )

        result_df.write.format("delta").mode("append").saveAsTable("noc_pm_documents_tessaract")
        print(f"[SUCCESS] {path}")

    except Exception as e:
        print(f"[FATAL ERROR] {path} -> {e}")

    

print("JOB FINISHED")

df = spark.table("noc_pm_documents_tessaract")
print("Total records saved:", df.count())
df.select("path", "is_scanned", "final_text").orderBy("path").show(10)


In [0]:
df = spark.table("noc_pm_documents_tessaract")
print("Total records saved:", df.count())
df.select("path", "is_scanned", "final_text").orderBy("path").show(150)

In [0]:
display(
    spark.table("noc_pm_documents_tessaract")
         .select("path", "is_scanned", "final_text")
         .orderBy("path")
)

In [0]:
# ======================= IMPORTS =======================
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import col, length
import fitz
from docx import Document
import os
import subprocess
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

print("JOB STARTED (TESSERACT OCR - SINGLE FILE MODE)")

# ======================= TARGET FILE =======================
TARGET_PATH = "dbfs:/FileStore/noc_pm_pdfs/0000005183.PDF"
print("Target file:", TARGET_PATH)

# ======================= READ FILE =======================
pdf_df = (
    spark.read.format("binaryFile")
    .load(TARGET_PATH)
    .select("path", "content")
)

print("Target file loaded")

# ======================= LOAD PROCESSED =======================
try:
    processed_df = (
        spark.table("noc_pm_documents_tessaract")
        .filter(col("final_text").isNotNull() & (length(col("final_text")) > 0))
        .select("path")
        .distinct()
    )
    print(f"Already processed files count: {processed_df.count()}")
except Exception:
    processed_df = spark.createDataFrame([], "path STRING")
    print("Table noc_pm_documents_tessaract does not exist")

# ======================= UNPROCESSED =======================
unprocessed_df = pdf_df.join(
    processed_df,
    on="path",
    how="left_anti"
)

count_unprocessed = unprocessed_df.count()
print(f"Unprocessed target files count: {count_unprocessed}")

if count_unprocessed == 0:
    print("File already processed — exiting job.")
    dbutils.notebook.exit("DONE")

# ======================= OCR HELPERS =======================
TMP_DIR = "/tmp/tesseract"
os.makedirs(TMP_DIR, exist_ok=True)

def convert_to_pdf(binary, path):
    ext = os.path.splitext(path.lower())[1]
    base = os.path.basename(path)
    src = f"{TMP_DIR}/{base}"
    pdf = f"{TMP_DIR}/{base}.pdf"

    with open(src, "wb") as f:
        f.write(binary)

    if ext == ".pdf":
        return src

    subprocess.run(
        [
            "libreoffice",
            "--headless",
            "--convert-to",
            "pdf",
            "--outdir",
            TMP_DIR,
            src
        ],
        check=True
    )
    return pdf

def ocr_tesseract(binary, path):
    print("[TESSERACT] OCR started")
    try:
        pdf_path = convert_to_pdf(binary, path)

        images = convert_from_path(
            pdf_path,
            dpi=300,
            output_folder=TMP_DIR
        )

        text = []
        for img in images:
            text.append(
                pytesseract.image_to_string(img, lang="eng")
            )

        final_text = "\n".join(text)
        print(f"[TESSERACT] Success ({len(final_text)} chars)")
        return final_text

    except Exception as e:
        print(f"[TESSERACT ERROR] {path} -> {e}")
        return ""

# ======================= TEXT EXTRACTORS =======================
def extract_text_from_pdf(binary, path):
    print("[FITZ] Extracting Text")
    try:
        doc = fitz.open(stream=binary, filetype="pdf")
        text = "\n".join(page.get_text() for page in doc)
        print(f"[FITZ] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[FITZ ERROR] {path} -> {e}")
        return ""

def extract_text_from_docx(binary, path):
    print("[DOCX] Extracting Text")
    try:
        tmp_path = f"/tmp/{os.path.basename(path)}"
        with open(tmp_path, "wb") as f:
            f.write(binary)

        doc = Document(tmp_path)
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
        print(f"[DOCX] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[DOCX ERROR] {path} -> {e}")
        return ""

def extract_text_from_plain(binary, path):
    try:
        text = binary.decode("utf-8", errors="ignore")
        print(f"[PLAIN] Success ({len(text)} chars)")
        return text
    except Exception:
        return ""

def extract_text(binary, path):
    ext = os.path.splitext(path.lower())[1]

    if ext == ".pdf":
        text = extract_text_from_pdf(binary, path)

    elif ext == ".docx":
        text = extract_text_from_docx(binary, path)

    elif ext == ".doc":
        print("[DOC] Legacy .doc → Tesseract OCR")
        return ocr_tesseract(binary, path)

    elif ext in {".txt", ".csv", ".md", ".rtf"}:
        text = extract_text_from_plain(binary, path)

    else:
        print(f"[INFO] Unsupported format {ext} → OCR")
        return ocr_tesseract(binary, path)

    if not text or len(text.strip()) < 200:
        print("[FALLBACK] Text too small → OCR")
        return ocr_tesseract(binary, path)

    return text

# ======================= WRITE =======================
schema = StructType([
    StructField("path", StringType(), True),
    StructField("final_text", StringType(), True),
    StructField("is_scanned", BooleanType(), True)
])

row = unprocessed_df.first()

path = row["path"]
binary = row["content"]

try:
    final_text = extract_text(binary, path)
    is_scanned = len(final_text.strip()) < 200

    result_df = spark.createDataFrame(
        [(path, final_text, is_scanned)],
        schema
    )

    result_df.write.format("delta").mode("append").saveAsTable("noc_pm_documents_tessaract")
    print(f"[SUCCESS] {path}")

except Exception as e:
    print(f"[FATAL ERROR] {path} -> {e}")

print("JOB FINISHED")

df = spark.table("noc_pm_documents_tessaract")
print("Total records saved:", df.count())

df.filter(col("path") == TARGET_PATH).show()


In [0]:
from pyspark.sql.functions import col
display(

    spark.table("noc_pm_documents_tessaract")
         .filter(col("path") == "dbfs:/FileStore/noc_pm_pdfs/0000005183.PDF")
         .select("path", "is_scanned", "final_text")
)

# OCR documents for required for experimentation using Tesseract

In [0]:
# ======================= IMPORTS =======================
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import col, length
import fitz
from docx import Document
import os
import subprocess
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

print("JOB STARTED (TESSERACT OCR)")

IN_PATH = "dbfs:/FileStore/accuracy_experiment/"
OUT_TABLE = "tesseract_accuracy"   # ✅ only one table

# ======================= READ FILES =======================
pdf_df = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(IN_PATH)
    .select("path", "content")
)
print("Files read from DBFS")

# ======================= LOAD PROCESSED =======================
try:
    processed_df = (
        spark.table(OUT_TABLE)
        .filter(col("final_text").isNotNull() & (length(col("final_text")) > 0))
        .select("path")
        .distinct()
    )
    print(f"Already processed files: {processed_df.count()}")
except Exception:
    processed_df = spark.createDataFrame([], "path STRING")
    print(f"Table {OUT_TABLE} does not exist (starting fresh)")

# ======================= UNPROCESSED =======================
unprocessed_df = pdf_df.join(processed_df, on="path", how="left_anti")
print(f"Unprocessed files count: {unprocessed_df.count()}")

# ======================= OCR HELPERS =======================
TMP_DIR = "/tmp/tesseract"
os.makedirs(TMP_DIR, exist_ok=True)

def convert_to_pdf(binary, path):
    ext = os.path.splitext(path.lower())[1]
    base = os.path.basename(path)
    src = f"{TMP_DIR}/{base}"
    pdf = f"{TMP_DIR}/{base}.pdf"

    with open(src, "wb") as f:
        f.write(binary)

    if ext == ".pdf":
        return src

    subprocess.run(
        ["libreoffice", "--headless", "--convert-to", "pdf", "--outdir", TMP_DIR, src],
        check=True
    )
    return pdf

def ocr_tesseract(binary, path):
    print("[TESSERACT] OCR started")
    try:
        pdf_path = convert_to_pdf(binary, path)

        images = convert_from_path(pdf_path, dpi=300, output_folder=TMP_DIR)

        text = []
        for img in images:
            text.append(pytesseract.image_to_string(img, lang="eng"))

        final_text = "\n".join(text)
        print(f"[TESSERACT] Success ({len(final_text)} chars)")
        return final_text

    except Exception as e:
        print(f"[TESSERACT ERROR] {path} -> {e}")
        return ""

# ======================= TEXT EXTRACTORS =======================
def extract_text_from_pdf(binary, path):
    print("[FITZ] Extracting Text")
    try:
        doc = fitz.open(stream=binary, filetype="pdf")
        text = "\n".join(page.get_text() for page in doc)
        print(f"[FITZ] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[FITZ ERROR] {path} -> {e}")
        return ""

def extract_text_from_docx(binary, path):
    print("[DOCX] Extracting Text")
    try:
        tmp_path = f"/tmp/{os.path.basename(path)}"
        with open(tmp_path, "wb") as f:
            f.write(binary)

        doc = Document(tmp_path)
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
        print(f"[DOCX] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[DOCX ERROR] {path} -> {e}")
        return ""

def extract_text_from_plain(binary, path):
    try:
        text = binary.decode("utf-8", errors="ignore")
        print(f"[PLAIN] Success ({len(text)} chars)")
        return text
    except Exception:
        return ""

def extract_text(binary, path):
    ext = os.path.splitext(path.lower())[1]

    if ext == ".pdf":
        text = extract_text_from_pdf(binary, path)

    elif ext == ".docx":
        text = extract_text_from_docx(binary, path)

    elif ext == ".doc":
        print("[DOC] Legacy .doc → Tesseract OCR")
        return ocr_tesseract(binary, path)

    elif ext in {".txt", ".csv", ".md", ".rtf", ".png"}:  # ✅ ".png" fixed
        # NOTE: if your pngs are images, you probably want OCR (not utf-8 decode).
        if ext == ".png":
            try:
                img_path = f"{TMP_DIR}/{os.path.basename(path)}"
                with open(img_path, "wb") as f:
                    f.write(binary)
                return pytesseract.image_to_string(Image.open(img_path), lang="eng")
            except Exception:
                return ""
        text = extract_text_from_plain(binary, path)

    else:
        print(f"[INFO] Unsupported format {ext} → OCR")
        return ocr_tesseract(binary, path)

    if not text or len(text.strip()) < 200:
        print("[FALLBACK] Text too small → OCR")
        return ocr_tesseract(binary, path)

    return text

# ======================= WRITE =======================
schema = StructType([
    StructField("path", StringType(), True),
    StructField("final_text", StringType(), True),
    StructField("is_scanned", BooleanType(), True)
])

rows = list(unprocessed_df.toLocalIterator())

for row in rows[:40]:
    path = row["path"]
    binary = row["content"]

    try:
        final_text = extract_text(binary, path)
        is_scanned = len(final_text.strip()) < 200

        result_df = spark.createDataFrame([(path, final_text, is_scanned)], schema)

        # ✅ write to the ONE table you want
        result_df.write.format("delta").mode("append").saveAsTable(OUT_TABLE)
        print(f"[SUCCESS] {path}")

    except Exception as e:
        print(f"[FATAL ERROR] {path} -> {e}")

print("JOB FINISHED")

df = spark.table(OUT_TABLE)
print("Total records saved:", df.count())
df.select("path", "is_scanned", "final_text").orderBy("path").show(10, truncate=False)


In [0]:
display(
    spark.table("tesseract_accuracy")
         .select("path", "is_scanned", "final_text")
         .orderBy("path")
)